In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from scipy.stats import multivariate_normal, norm

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "snp500.csv", index_col=1).reset_index()
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "china_transformed.csv", index_col=1).reset_index()
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "emerging_markets.csv", index_col=1)
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "gold.csv", index_col=1)
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "india.csv", index_col=1)
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "msci_europe.csv", index_col=1)
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "small_cap_europe.csv", index_col=1)

labels = ["snp", "china", "em", "gold", "india", "mscieurope", "smallcapeurope"]
dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]

def transform_date(date):
    parts = date.split(' ')
    if len(parts) == 1:
        return date
    return parts[0]

for df in dfs:
    df["Date"] = df["Date"].apply(transform_date)

china

date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
filtered_dfs = [df[df["Date"].isin(common_dates)] for df in dfs]


In [ ]:
from scipy.stats import rankdata

N = len(filtered_dfs)
T = len(filtered_dfs[0]) - 1

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))

for i, (label, df) in enumerate(zip(labels, filtered_dfs)):
    returns = df["Close"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)

U = np.zeros_like(returns_all) # (T,N)
for i in range(N):
    ranks = rankdata(returns_all[:, i], method='average') # compute ranks to throw away
    # things like units, magnitudes etc
    # for copula correlations: we purely want rank-based information
    U[:, i] = ranks / (T + 1)

def inverse_cdf(u, sorted_returns):
    n = len(sorted_returns)
    idx = np.minimum((u * n).astype(int), n - 1)
    return sorted_returns[idx]
    
Z = norm.ppf(U) # (T, N) (performed elementwise)
cov_copula = np.cov(Z, rowvar=False) # (N, N)
num_samples = 100

Z_samples = multivariate_normal.rvs(
    mean=np.zeros(N),
    cov=cov_copula,
    size=num_samples
)
U_samples = norm.cdf(Z_samples) # (T, N)
X_samples = np.zeros_like(U_samples) # (T, N)

for i in range(N):
    X_samples[:, i] = inverse_cdf(
        U_samples[:, i],
        sorted_returns_all[:, i]
    )

